# lb-whisper: fine-tune Whisper large-v3-turbo on Luxembourgish (FLEURS lb_lu)

Runtime > Change runtime type > T4 GPU. The notebook only runs the scripts of the repo, in order. The 3-epoch run took about 2.1 h on a T4.

In [ ]:
!nvidia-smi -L
!git clone https://github.com/Sitraka17/lb-whisper.git
%cd lb-whisper/finetune
!pip install -q -r requirements.txt
!apt-get -qq install -y ffmpeg > /dev/null

## Hugging Face login
The token comes from the Colab secret `HF_TOKEN` (key icon, notebook access on, write access). Only needed to publish; the data needs no login.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

## Drive and environment
Checkpoints go to Google Drive so a disconnect does not lose them. Colab ships torchao 0.10, too old for recent peft: it is removed.

In [ ]:
!pip uninstall -y -q torchao
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/lb-whisper/runs
!ln -sfn /content/drive/MyDrive/lb-whisper/runs runs

## 1. Data
Google FLEURS, Luxembourgish (`lb_lu`), CC-BY-4.0, downloaded from the Hub without login.

In [ ]:
!python prepare_data.py --out data/fleurs_lb
!cat data/fleurs_lb/stats.json

## 2. Train
Add `--resume` after a disconnect. For a quick check first, add `--max-train-samples 300`.

In [ ]:
!python train.py --config configs/turbo-lora.yaml

## 3. Export and evaluate on the full test split

In [ ]:
!python export.py --run runs/turbo-lora
!python evaluate.py --model runs/turbo-lora/merged --data data/fleurs_lb --split test

## 4. Benchmark
First 300 test utterances, clean and 16 kbit/s Opus. faster-whisper (for the ZLS model) needs the cuBLAS and cuDNN 9 libraries on the library path.

In [ ]:
!pip install -q nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"
import glob, os
os.environ["LD_LIBRARY_PATH"] = ":".join(glob.glob("/usr/local/lib/python3*/dist-packages/nvidia/*/lib") + [os.environ.get("LD_LIBRARY_PATH", "")])
!python benchmark.py --data data/fleurs_lb --max-samples 300 --models \
    openai/whisper-large-v3-turbo openai/whisper-small \
    unilux/whisper-medium-v1-luxembourgish ZLSCompLing/Whisper-Large-V3-Turbo-Luxembourgish \
    runs/turbo-lora/merged
!cat results/benchmark.md

## 5. Publish
`fill_card.py` writes the model card from the result files and stops if anything is missing.

In [ ]:
!python fill_card.py --run runs/turbo-lora --data data/fleurs_lb --out results/model_card.md
!python export.py --run runs/turbo-lora --card results/model_card.md \
    --push SitrakaForler/whisper-large-v3-turbo-lb --push-ct2 SitrakaForler/whisper-large-v3-turbo-lb-ct2